In [ ]:
import pickle

with open("../models/job_tfidf_vectorizer.pkl", "wb") as f:
    pickle.dump(job_tfidf, f)

with open("../data/processed/job_vectors.pkl", "wb") as f:
    pickle.dump(job_vectors, f)

job_corpus.to_csv("../data/processed/job_corpus_final.csv", index=False)

print("Saved recommender artifacts.")

In [ ]:
print(
    f"\n{mismatch_count} out of {resumes['Category'].nunique()} sampled categories show possible label/content mismatch"
)

In [ ]:
mismatch_count = 0

for cat in resumes["Category"].unique():

    sample = resumes[
        resumes["Category"]==cat
    ].iloc[0]

    cat_word = cat.lower().split("-")[0]

    if cat_word not in sample["Resume"].lower():

        mismatch_count += 1

        print(f"Possible mismatch - Category: {cat}")

In [ ]:
print("Does it contain 'chef'?:","chef" in chef_resume["Resume"].lower())

print("Does it contain 'cook'?:","cook" in chef_resume["Resume"].lower())

print("Does it contain 'culinary'?:","culinary" in chef_resume["Resume"].lower())

In [ ]:
print("Raw resume text (first 500 chars):")
print(chef_resume["Resume"][:500])

In [ ]:
chef_resume = resumes[
    resumes["Category"]=="Data Science" # Changed from 'CHEF' to 'Data Science'
].iloc[0]

In [ ]:
test_recs = recommend_jobs(
    "experienced chef cook culinary kitchen restaurant executive chef",
    top_n=5
)

print("Clean chef query results:")

print(test_recs[["title","match_score"]].to_string(index=False))

In [ ]:
for category, keywords in keywords_to_check.items():

    pattern = "|".join(keywords)

    matches = job_corpus[
        job_corpus["title"].str.lower().str.contains(pattern, na=False)
    ]

    print(f"{category}: {len(matches)} matching job titles found in corpus")

    if len(matches) > 0:
        print(matches["title"].head(5).tolist())

    print()

In [ ]:
keywords_to_check = {
    "CHEF": ["chef", "cook", "culinary", "kitchen"],
    "AVIATION": ["pilot", "aviation", "aircraft", "flight"]
}

In [ ]:
for cat in categories_to_check:

    sample_row = resumes[resumes["Category"] == cat].iloc[0]

    print("\n" + "="*50)
    print(f"Resume Category: {cat}")

    recs = recommend_jobs(sample_row["Resume"], top_n=5)

    print(recs[["title","match_score"]].to_string(index=False))

In [ ]:
categories_to_check = [
    "Data Science",
    "Advocate",
    "HR",
    "Python Developer"
]

In [ ]:
score = precision_at_k(k=10, n_samples=50)

print(f"Precision@10 (category-word proxy): {round(score,4)}")

In [ ]:
def precision_at_k(k=10, n_samples=50):
    """
    For a sample of resumes, check what fraction of top-K recommended jobs
    have a title containing a word that overlaps with the resume's known category.
    This is a proxy since we don't have ground-truth job-resume pairs.
    """
    sample = resumes.sample(n=n_samples, random_state=42)
    hits = 0
    total = 0

    for _, row in sample.iterrows():
        category_words = row["Category"].lower().replace("-", " ").split()
        recs = recommend_jobs(row["Resume"], top_n=k)

        for title in recs["title"]:
            total += 1
            title_lower = str(title).lower()
            if any(word in title_lower for word in category_words):
                hits += 1

    return hits / total if total > 0 else 0

score = precision_at_k(k=10, n_samples=50)
print(f"Precision@10 (category-word proxy): {round(score, 4)}")

In [ ]:
from sklearn.metrics.pairwise import cosine_similarity

def recommend_jobs(resume_text, top_n=10):
    # Clean the resume text the same way the job corpus was cleaned
    cleaned = clean_text(resume_text)

    # Transform using the SAME fitted vectorizer (not fit again!)
    resume_vector = job_tfidf.transform([cleaned])

    # Compute cosine similarity between this resume and every job
    similarities = cosine_similarity(resume_vector, job_vectors).flatten()

    # Get indices of top-N most similar jobs
    top_indices = similarities.argsort()[::-1][:top_n]

    results = job_corpus.iloc[top_indices][["title", "company", "location", "source"]].copy()
    results["match_score"] = similarities[top_indices]
    return results.reset_index(drop=True)

# We need clean_text function again since this is a new notebook
import re
def clean_text(text):
    if not isinstance(text, str):
        return ""
    text = text.lower()
    text = re.sub(r"[^a-z0-9\s]", " ", text)
    text = re.sub(r"\s+", " ", text).strip()
    return text
resumes = pd.read_csv("../data/processed/resumes_clean.csv")
sample_resume = resumes["Resume"].iloc[0]

print("Testing recommender on this resume's category:", resumes["Category"].iloc[0])
print("\nTop 10 recommended jobs:")
recommend_jobs(sample_resume, top_n=10)

In [ ]:
import pandas as pd
from sklearn.feature_extraction.text import TfidfVectorizer

job_corpus = pd.read_csv("../data/processed/job_corpus_clean.csv")
job_corpus = job_corpus.dropna(subset=["clean_text"]).reset_index(drop=True)

print("Job corpus shape:", job_corpus.shape)

# Fit a fresh TF-IDF vectorizer on the job corpus (separate from the resume classifier's vectorizer)
job_tfidf = TfidfVectorizer(max_features=10000, stop_words="english")
job_vectors = job_tfidf.fit_transform(job_corpus["clean_text"])

print("Job vectors shape:", job_vectors.shape)